# Análisis de customers.csv

**Autor:** Daniel Guzmán  
**Fecha:** 2026-04-23  
**Entorno:** Databricks

In [0]:
import time
from pyspark.sql import functions as F

In [0]:
catalog = "workspace"
schema = "default"
volume = "customers_files_daniel"

path_volume = f"/Volumes/{catalog}/{schema}/{volume}"

print(path_volume)

In [0]:
inicio = time.time()

df_csv = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{path_volume}/customers.csv")
)

total_registros = df_csv.count()
fin = time.time()

print(f"Tiempo de lectura: {fin - inicio:.4f} segundos")
print(f"Registros: {total_registros}")
print(f"Columnas: {df_csv.columns}")

In [0]:
display(df_csv.limit(5))

In [0]:
print("Tipos de datos:")
for col_name, dtype in df_csv.dtypes:
    print(f"{col_name}: {dtype}")

In [0]:
print("Shape:")
print(f"Filas: {df_csv.count()}")
print(f"Columnas: {len(df_csv.columns)}")

In [0]:
df_csv.printSchema()

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when

nulos_df = df_csv.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_csv.columns
])

display(nulos_df)

In [0]:
for c in df_csv.columns:
    print(f"{c}: {df_csv.select(c).distinct().count()} valores únicos")

In [0]:
display(df_csv.describe())

In [0]:
1. ¿Cuántos clientes hay por país?

clientes_por_pais = (
    df_csv.groupBy("Country")
    .count()
    .withColumnRenamed("count", "total_clientes")
    .orderBy(F.col("total_clientes").desc())
)

display(clientes_por_pais)

In [0]:

display(clientes_por_pais.limit(5))

In [0]:
empresas_distintas = df_csv.select("Company").distinct().count()
print(f"Empresas distintas: {empresas_distintas}")

In [0]:
nombres_frecuentes = (
    df_csv.withColumn("Nombre Completo", F.concat_ws(" ", F.col("First Name"), F.col("Last Name")))
    .groupBy("Nombre Completo")
    .count()
    .orderBy(F.col("count").desc())
)

display(nombres_frecuentes.limit(1))

In [0]:
clientes_sin_ciudad = df_csv.filter(
    F.col("City").isNull() | (F.trim(F.col("City")) == "")
).count()

print(f"Clientes sin ciudad registrada: {clientes_sin_ciudad}")

## Reflexión sobre el formato .csv

- Tiempo de lectura registrado: 13.8104 segundos
- Tamaño del archivo: 1.72 MB aprox.
- Fue sencillo de leer con PySpark en Databricks.
- Como ventaja, CSV es muy común y fácil de compartir. Como desventaja, no conserva tipos de datos de forma nativa y suele ser menos eficiente.
- Usaría CSV para intercambios simples de datos, cargas iniciales o archivos que deban ser entendidos por muchas herramientas.
